In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import ResNetForImageClassification,MobileNetV2ForImageClassification
from PIL import Image
import os
from pathlib import Path
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.optim import lr_scheduler
import random
import cv2


In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-3

In [ ]:
def get_transforms():
    # ✅ BACK TO RGB TRANSFORMS - NO MORE HSV
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        # rgb_to_hsv_transform(),  # ❌ REMOVED - no more HSV conversion
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ✅ Standard RGB ImageNet normalization
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        # rgb_to_hsv_transform(),  # ❌ REMOVED - no more HSV conversion
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ✅ Standard RGB ImageNet normalization
    ])
    
    return train_transform, val_transform

In [4]:
def setup_finetuning(model, freeze_backbone=True):
  
    if freeze_backbone:
        for param in model.mobilenet_v2.parameters():
            param.requires_grad = False
        
        # Só treina o classificador final
        for param in model.classifier.parameters():
            param.requires_grad = True
            
        print(f"Parâmetros treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    else:
        print(" Fine-tuning completo com learning rates diferentes")
        
    return model

In [ ]:
def test_on_complete_dataset():
    """
    Testa o melhor modelo HSV em TODAS as imagens do dataset original usando labels.txt
    """
    
    # ✨ Check for the best available model (fold_0 or fold_1)
    available_models = []
    for fold_idx in range(5):
        model_path = Path(f"results/best_model_fold_{fold_idx}.pth")
        if model_path.exists():
            available_models.append((fold_idx, model_path))
    
    if not available_models:
        print("❌ Nenhum modelo encontrado!")
        return None
    
    # Use the first available model
    fold_idx, model_path = available_models[0]
    print(f"Carregando modelo : {model_path.name}")
    
    model = MobileNetV2ForImageClassification.from_pretrained(
        "google/mobilenet_v2_1.0_224", 
        num_labels=3,
        ignore_mismatched_sizes=True
    )
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model = setup_finetuning(model, freeze_backbone=True)
    model.to(DEVICE)
    model.eval()
    
    # Dataset original
    original_dataset = Path(r"C:\Users\User\Documents\IA_PROJECT\Archives\files_webot\IA_20252\controllers\dataset_generator\dataset_cubes_only")
    
    if not original_dataset.exists():
        print(f"Dataset original não encontrado: {original_dataset}")
        return None
    
    # LER LABELS.TXT E CRIAR PARES (imagem, label)
    labels_file = original_dataset / "labels.txt"
    if not labels_file.exists():
        print(f"❌ Arquivo labels.txt não encontrado: {labels_file}")
        return None
    
    print(f"📄 Carregando labels de: {labels_file}")
    
    pairs = []
    label_counts = {"red": 0, "green": 0, "blue": 0}
    
    with open(labels_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Formato: img_000001.png red
            parts = line.split()
            if len(parts) >= 2:
                img_name = parts[0]
                label_name = parts[1]
                
                img_path = original_dataset / img_name
                if img_path.exists() and label_name in ["red", "green", "blue"]:
                    pairs.append((img_path, label_name))
                    label_counts[label_name] += 1
    
    if not pairs:
        print("❌ Nenhum par (imagem, label) válido!")
        return None
    
    print(f"📊 Dataset carregado:")
    for color, count in label_counts.items():
        print(f"  ✅ {color}: {count} imagens")
    print(f"📊 Total: {len(pairs)} imagens")
    
    _, test_transform = get_transforms()
    
    # Mapear labels para índices
    label_to_idx = {"red": 0, "green": 1, "blue": 2}
    
    results = {}
    total_correct = 0
    total_images = len(pairs)
    class_correct = {"red": 0, "green": 0, "blue": 0}
    class_total = {"red": 0, "green": 0, "blue": 0}
    
    print("\n🧪 TESTE NO DATASET COMPLETO (HSV MODEL):")
    
    # Processar todas as imagens
    for img_path, true_label in tqdm(pairs, desc="Testando imagens"):
        try:
            image = Image.open(img_path).convert('RGB')
            image_tensor = test_transform(image).unsqueeze(0).to(DEVICE)
            
            with torch.no_grad():
                outputs = model(image_tensor).logits
                predicted_idx = torch.max(outputs, 1)[1].item()
                predicted_label = list(label_to_idx.keys())[predicted_idx]
                
                # Contabilizar por classe
                class_total[true_label] += 1
                
                if predicted_label == true_label:
                    total_correct += 1
                    class_correct[true_label] += 1
                        
        except Exception as e:
            print(f"❌ Erro {img_path}: {e}")
            continue
    
    # Calcular resultados por classe
    print(f"\n📊 RESULTADOS POR CLASSE (HSV MODEL):")
    for color in ["red", "green", "blue"]:
        if class_total[color] > 0:
            accuracy = (class_correct[color] / class_total[color]) * 100
            results[color] = {
                'correct': class_correct[color],
                'total': class_total[color],
                'accuracy': accuracy
            }
            print(f"  🎨 {color}: {class_correct[color]}/{class_total[color]} = {accuracy:.2f}%")
        else:
            results[color] = {'correct': 0, 'total': 0, 'accuracy': 0}
    
    overall_accuracy = (total_correct / total_images) * 100 if total_images > 0 else 0
    
    print(f"\n🏆 RESULTADOS FINAIS (HSV vs RGB COMPARISON):")
    print(f"📊 HSV Model Accuracy: {overall_accuracy:.2f}%")
    print(f"📊 Total: {total_correct}/{total_images}")
    print(f"📊 HSV Validation: 97.53% (average from training)")
    
    # ✨ Compare with previous RGB results
    print(f"\n📈 IMPROVEMENT ANALYSIS:")
    rgb_results = {"red": 99.47, "green": 99.75, "blue": 91.61, "overall": 97.47}
    
    for color in ["red", "green", "blue"]:
        if color in results:
            improvement = results[color]['accuracy'] - rgb_results[color]
            print(f"  🎨 {color}: {results[color]['accuracy']:.2f}% (RGB: {rgb_results[color]:.2f}%) → {improvement:+.2f}pp")
    
    overall_improvement = overall_accuracy - rgb_results['overall']
    print(f"  📊 Overall: {overall_accuracy:.2f}% (RGB: {rgb_results['overall']:.2f}%) → {overall_improvement:+.2f}pp")
    
    # Salvar resultados
    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)
    
    final_test_results = {
        'model_type': 'HSV',
        'model_used': str(model_path),
        'overall_accuracy': overall_accuracy,
        'total_correct': total_correct,
        'total_images': total_images,
        'results_by_color': results,
        'hsv_validation_acc': 97.53,
        'rgb_comparison': rgb_results,
        'improvement': {
            'overall': overall_improvement,
            'per_color': {color: results[color]['accuracy'] - rgb_results[color] 
                         for color in ["red", "green", "blue"] if color in results}
        },
        'dataset_source': 'labels.txt'
    }
    
    # ✨ Save with HSV suffix
    with open(results_dir / "hsv_test_results.json", "w") as f:
        json.dump(final_test_results, f, indent=2)
    
    print(f"\n Resultados HSV salvos em: results/hsv_test_results.json")
    
    return final_test_results

test_on_complete_dataset()

📥 Carregando modelo HSV: best_model_fold_0.pth


Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([3, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parâmetros treináveis: 3,843
📄 Carregando labels de: C:\Users\User\Documents\IA_PROJECT\Archives\files_webot\IA_20252\controllers\dataset_generator\dataset_cubes_only\labels.txt
📊 Dataset carregado:
  ✅ red: 2304 imagens
  ✅ green: 3840 imagens
  ✅ blue: 5376 imagens
📊 Total: 11520 imagens

🧪 TESTE NO DATASET COMPLETO (HSV MODEL):


Testando imagens: 100%|██████████| 11520/11520 [02:27<00:00, 78.20it/s]


📊 RESULTADOS POR CLASSE (HSV MODEL):
  🎨 red: 2289/2304 = 99.35%
  🎨 green: 3591/3840 = 93.52%
  🎨 blue: 5330/5376 = 99.14%

🏆 RESULTADOS FINAIS (HSV vs RGB COMPARISON):
📊 HSV Model Accuracy: 97.31%
📊 Total: 11210/11520
📊 HSV Validation: 97.53% (average from training)

📈 IMPROVEMENT ANALYSIS:
  🎨 red: 99.35% (RGB: 99.47%) → -0.12pp
  🎨 green: 93.52% (RGB: 99.75%) → -6.23pp
  🎨 blue: 99.14% (RGB: 91.61%) → +7.53pp
  📊 Overall: 97.31% (RGB: 97.47%) → -0.16pp

 Resultados HSV salvos em: results/hsv_test_results.json


{'model_type': 'HSV',
 'model_used': 'results\\best_model_fold_0.pth',
 'overall_accuracy': 97.30902777777779,
 'total_correct': 11210,
 'total_images': 11520,
 'results_by_color': {'red': {'correct': 2289,
   'total': 2304,
   'accuracy': 99.34895833333334},
  'green': {'correct': 3591, 'total': 3840, 'accuracy': 93.515625},
  'blue': {'correct': 5330, 'total': 5376, 'accuracy': 99.14434523809523}},
 'hsv_validation_acc': 97.53,
 'rgb_comparison': {'red': 99.47,
  'green': 99.75,
  'blue': 91.61,
  'overall': 97.47},
 'improvement': {'overall': -0.1609722222222132,
  'per_color': {'red': -0.12104166666665606,
   'green': -6.234375,
   'blue': 7.534345238095227}},
 'dataset_source': 'labels.txt'}

In [6]:

def test_single_image(model_path, image_path):
    model = MobileNetV2ForImageClassification.from_pretrained(
        "google/mobilenet_v2_1.0_224", 
        num_labels=3,
        ignore_mismatched_sizes=True
    )
    model.load_state_dict(torch.load(model_path))
    model.to(DEVICE)
    model.eval()
    
    _, val_transform = get_transforms()
    image = Image.open(image_path).convert('RGB')
    image_tensor = val_transform(image).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(image_tensor).logits
        predicted = torch.max(outputs, 1)[1].item()
        probabilities = torch.softmax(outputs, dim=1)[0]
    
    classes = ['red', 'green', 'blue']
    predicted_class = classes[predicted]
    confidence = probabilities[predicted].item() * 100
    
    print(f"Predição: {predicted_class} ({confidence:.2f}%)")
    
    for i, cls in enumerate(classes):
        prob = probabilities[i].item() * 100
        print(f"  {cls}: {prob:.2f}%")



In [ ]:
def test_individual_folds():
    """Test each fold's model on its own test set"""
    
    folds_dir = Path("cubes_classification_folds")
    results_dir = Path("results")
    
    fold_test_results = []
    
    print("TESTING INDIVIDUAL FOLDS:")
    
    for fold_idx in range(5):  # Test all 5 folds
        model_path = results_dir / f"best_model_fold_{fold_idx}.pth"
        if not model_path.exists():
            print(f"❌ Model for fold {fold_idx} not found")
            continue
            
        print(f"\n🔬 Testing Fold {fold_idx}...")
        
        # Load model
        model = MobileNetV2ForImageClassification.from_pretrained(
            "google/mobilenet_v2_1.0_224", 
            num_labels=3,
            ignore_mismatched_sizes=True
        )
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        model = setup_finetuning(model, freeze_backbone=True)
        model.to(DEVICE)
        model.eval()
        
        # Test on fold's test set
        test_path = folds_dir / f"fold_{fold_idx}" / "test"
        if not test_path.exists():
            print(f"❌ Test set for fold {fold_idx} not found")
            continue
            
        fold_result = test_fold_on_test_set(model, test_path, fold_idx)
        if fold_result:
            fold_test_results.append(fold_result)
    
    # Analyze results across folds
    analyze_fold_results(fold_test_results)
    return fold_test_results

def test_fold_on_test_set(model, test_path, fold_idx):
    """Test a specific fold's model on its test set"""
    
    _, test_transform = get_transforms()
    
    class_results = {}
    total_correct = 0
    total_images = 0
    
    print(f"  📁 Testing on {test_path}")
    
    for color_idx, color in enumerate(['red', 'green', 'blue']):
        color_dir = test_path / color
        if not color_dir.exists():
            continue
            
        images = list(color_dir.glob("*.png"))
        correct = 0
        
        for img_path in images:
            try:
                image = Image.open(img_path).convert('RGB')
                image_tensor = test_transform(image).unsqueeze(0).to(DEVICE)
                
                with torch.no_grad():
                    outputs = model(image_tensor).logits
                    predicted = torch.max(outputs, 1)[1].item()
                    
                    if predicted == color_idx:
                        correct += 1
                        
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                continue
        
        accuracy = (correct / len(images)) * 100 if len(images) > 0 else 0
        class_results[color] = {
            'correct': correct,
            'total': len(images),
            'accuracy': accuracy
        }
        
        total_correct += correct
        total_images += len(images)
        
        print(f"    🎨 {color}: {correct}/{len(images)} = {accuracy:.2f}%")
    
    overall_accuracy = (total_correct / total_images) * 100 if total_images > 0 else 0
    print(f"    📊 Fold {fold_idx} Test Accuracy: {overall_accuracy:.2f}%")
    
    return {
        'fold': fold_idx,
        'overall_accuracy': overall_accuracy,
        'total_correct': total_correct,
        'total_images': total_images,
        'class_results': class_results
    }

def analyze_fold_results(fold_results):
    """Analyze results across all folds"""

    
    print(f"\n🏆 CROSS-FOLD ANALYSIS:")
    
    # Overall accuracies
    overall_accs = [r['overall_accuracy'] for r in fold_results]
    mean_acc = np.mean(overall_accs)
    std_acc = np.std(overall_accs)
    
    print(f"Average Test Accuracy: {mean_acc:.2f} ± {std_acc:.2f}%")
    
    # Per-color analysis
    for color in ['red', 'green', 'blue']:
        color_accs = []
        for result in fold_results:
            if color in result['class_results']:
                color_accs.append(result['class_results'][color]['accuracy'])
        
        if color_accs:
            mean_color = np.mean(color_accs)
            std_color = np.std(color_accs)
            print(f"🎨 {color.capitalize()}: {mean_color:.2f} ± {std_color:.2f}%")
    
    # Save detailed results
    with open("results/fold_test_analysis.json", "w") as f:
        json.dump({
            'summary': {
                'mean_accuracy': mean_acc,
                'std_accuracy': std_acc,
                'num_folds_tested': len(fold_results)
            },
            'detailed_results': fold_results
        }, f, indent=2)
    
    print(f"\nDetailed results saved to results/fold_test_analysis.json")

# Run the analysis
test_individual_folds()

TESTING INDIVIDUAL FOLDS:

🔬 Testing Fold 0...


Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([3, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parâmetros treináveis: 3,843
  📁 Testing on cubes_classification_folds\fold_0\test
    🎨 red: 349/388 = 89.95%
    🎨 green: 283/388 = 72.94%
    🎨 blue: 348/388 = 89.69%
    📊 Fold 0 Test Accuracy: 84.19%

🔬 Testing Fold 1...


Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([3, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parâmetros treináveis: 3,843
  📁 Testing on cubes_classification_folds\fold_1\test
    🎨 red: 379/388 = 97.68%
    🎨 green: 286/388 = 73.71%
    🎨 blue: 329/388 = 84.79%
    📊 Fold 1 Test Accuracy: 85.40%

🔬 Testing Fold 2...


Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([3, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parâmetros treináveis: 3,843
  📁 Testing on cubes_classification_folds\fold_2\test
    🎨 red: 0/388 = 0.00%
    🎨 green: 387/388 = 99.74%
    🎨 blue: 0/388 = 0.00%
    📊 Fold 2 Test Accuracy: 33.25%
❌ Model for fold 3 not found
❌ Model for fold 4 not found

🏆 CROSS-FOLD ANALYSIS:
Average Test Accuracy: 67.61 ± 24.30%
🎨 Red: 62.54 ± 44.34%
🎨 Green: 82.13 ± 12.46%
🎨 Blue: 58.16 ± 41.17%

Detailed results saved to results/fold_test_analysis.json


[{'fold': 0,
  'overall_accuracy': 84.19243986254295,
  'total_correct': 980,
  'total_images': 1164,
  'class_results': {'red': {'correct': 349,
    'total': 388,
    'accuracy': 89.94845360824742},
   'green': {'correct': 283, 'total': 388, 'accuracy': 72.9381443298969},
   'blue': {'correct': 348, 'total': 388, 'accuracy': 89.69072164948454}}},
 {'fold': 1,
  'overall_accuracy': 85.39518900343643,
  'total_correct': 994,
  'total_images': 1164,
  'class_results': {'red': {'correct': 379,
    'total': 388,
    'accuracy': 97.68041237113401},
   'green': {'correct': 286, 'total': 388, 'accuracy': 73.71134020618557},
   'blue': {'correct': 329, 'total': 388, 'accuracy': 84.79381443298969}}},
 {'fold': 2,
  'overall_accuracy': 33.24742268041237,
  'total_correct': 387,
  'total_images': 1164,
  'class_results': {'red': {'correct': 0, 'total': 388, 'accuracy': 0.0},
   'green': {'correct': 387, 'total': 388, 'accuracy': 99.74226804123711},
   'blue': {'correct': 0, 'total': 388, 'accurac

: 